In [ ]:
# ==================================================
# contact_frequency_analysis_na.py
#
# Purpose:
#   Calculate contact frequency between TO ligand
#   and each nucleic acid residue.
#
# Input:
#   PSF topology file
#   DCD trajectory file
#
# Output:
#   Contact frequency bar plot
#
# Method:
#   For each residue:
#       calculate minimum distance to TO ligand
#       count frames with distance < cutoff
#       contact frequency = contact frames / total frames
#
# Notes:
#   Distance unit in MDTraj = nm
# ==================================================

import mdtraj as md
import numpy as np
import matplotlib.pyplot as plt


# ==================================================
# Part 1. User Settings
# ==================================================
#
# Most users only need to modify this section.
#
# ==================================================

# Simulation systems

simulations_data = {

    "606": {
        "topo": "/dicos_ui_home/chrysaliso/G4/TO_center_ion_na_1109/606/step3_input.psf",
        "traj": "/ceph/sharedfs/work/MYTLab/asher/center_ion_project/na/606_na/G4_TO_center_606.dcd",
        "label": "Docking score = 6.06",
        "color": "royalblue",
    },

    "687": {
        "topo": "/dicos_ui_home/chrysaliso/G4/TO_center_ion_na_1109/687/step3_input.psf",
        "traj": "/ceph/sharedfs/work/MYTLab/asher/center_ion_project/na/687_na/G4_TO_center_687.dcd",
        "label": "Docking score = 6.87",
        "color": "seagreen",
    },

    "688": {
        "topo": "/dicos_ui_home/chrysaliso/G4/TO_center_ion_na_1109/688/step3_input.psf",
        "traj": "/ceph/sharedfs/work/MYTLab/asher/center_ion_project/na/688_na/G4_TO_center_688.dcd",
        "label": "Docking score = 6.88",
        "color": "darkorange",
    },

    "688_2": {
        "topo": "/dicos_ui_home/chrysaliso/G4/TO_center_ion_na_1109/688_2/step3_input.psf",
        "traj": "/ceph/sharedfs/work/MYTLab/asher/center_ion_project/na/688_na_2/G4_TO_center_688_2.dcd",
        "label": "Docking score = 6.88_2",
        "color": "crimson",
    },

    "721": {
        "topo": "/dicos_ui_home/chrysaliso/G4/TO_center_ion_na_1109/721/step3_input.psf",
        "traj": "/ceph/sharedfs/work/MYTLab/asher/center_ion_project/na/721_na/G4_TO_center_721.dcd",
        "label": "Docking score = 7.21",
        "color": "purple",
    },

}


# Ligand residue name
#
# Check this name in PSF/PDB.
# For TO system, ligand residue name is usually TOG.

LIGAND_RESNAME = "TOG"


# Contact cutoff
#
# MDTraj distance unit = nm
# 0.5 nm = 5 Å

CONTACT_CUTOFF_NM = 0.5


# Trajectory stride
#
# stride = 10 means reading one frame every 10 frames.
# Increase stride to reduce memory usage and speed up analysis.

TRAJ_STRIDE = 10


# Nucleic acid residue names

NUCLEIC_ACID_RESNAMES = {
    "A", "C", "G", "T",
    "DA", "DC", "DG", "DT",
    "ADE", "CYT", "GUA", "THY",
}


# G-quartet layer residue numbers
#
# These residue numbers are used only for reordering
# and background grouping in the final plot.

G_QUARTET_LAYER1_RESSEQS = {2, 8, 14, 20}
G_QUARTET_LAYER2_RESSEQS = {3, 9, 15, 21}
G_QUARTET_LAYER3_RESSEQS = {4, 10, 16, 22}


# Figure settings

FIG_SIZE = (24, 10)
FIG_DPI = 150

Y_MIN = 0
Y_MAX = 1.1


# ==================================================
# Part 2. Helper Function
# ==================================================

def split_residue_label(label):
    """
    Split residue label into residue name and residue number.

    Example:
        DG12 -> DG, 12
        A5   -> A, 5
    """

    residue_name = "".join(
        filter(str.isalpha, label)
    )

    residue_number = int(
        "".join(filter(str.isdigit, label))
    )

    return residue_name, residue_number


def reorder_residue_labels(residue_labels):
    """
    Reorder residues into:
        G-quartet layer 1
        G-quartet layer 2
        G-quartet layer 3
        A/T-rich residues
        other residues
    """

    g_layer1_labels = []
    g_layer2_labels = []
    g_layer3_labels = []
    at_rich_labels = []
    other_labels = []

    for label in residue_labels:

        residue_name, residue_number = split_residue_label(
            label
        )

        is_guanine = residue_name in {
            "G",
            "DG",
            "GUA",
        }

        is_at_residue = residue_name in {
            "A",
            "T",
            "DA",
            "DT",
            "ADE",
            "THY",
        }

        if (
            is_guanine
            and residue_number in G_QUARTET_LAYER1_RESSEQS
        ):

            g_layer1_labels.append(label)

        elif (
            is_guanine
            and residue_number in G_QUARTET_LAYER2_RESSEQS
        ):

            g_layer2_labels.append(label)

        elif (
            is_guanine
            and residue_number in G_QUARTET_LAYER3_RESSEQS
        ):

            g_layer3_labels.append(label)

        elif is_at_residue:

            at_rich_labels.append(label)

        else:

            other_labels.append(label)

    reordered_labels = (
        g_layer1_labels
        + g_layer2_labels
        + g_layer3_labels
        + at_rich_labels
        + other_labels
    )

    group_sizes = {
        "g1": len(g_layer1_labels),
        "g2": len(g_layer2_labels),
        "g3": len(g_layer3_labels),
        "at": len(at_rich_labels),
        "other": len(other_labels),
    }

    return reordered_labels, group_sizes


def calculate_contact_frequency(
    traj,
    nucleic_residues,
    ligand_residue,
    cutoff_nm,
):
    """
    Calculate contact frequency between each nucleic acid
    residue and the ligand.

    Contact frequency:
        number of frames with minimum distance < cutoff
        divided by total number of frames
    """

    contact_frequencies = []
    residue_labels = []

    for residue in nucleic_residues:

        residue_ligand_pair = [
            [
                residue.index,
                ligand_residue.index,
            ]
        ]

        min_distance = md.compute_contacts(
            traj,
            contacts=residue_ligand_pair,
            scheme="closest",
        )[0]

        frequency = np.mean(
            min_distance < cutoff_nm
        )

        contact_frequencies.append(
            frequency
        )

        residue_labels.append(
            f"{residue.name}{residue.resSeq}"
        )

    return residue_labels, contact_frequencies


# ==================================================
# Part 3. Calculate Contact Frequency
# ==================================================

results = {}
reordered_labels = None
group_sizes = None

print("====================================")
print("Contact frequency analysis started")
print(f"Ligand residue name: {LIGAND_RESNAME}")
print(f"Contact cutoff: {CONTACT_CUTOFF_NM} nm")
print(f"Trajectory stride: {TRAJ_STRIDE}")
print("====================================")


for system_name, system_info in simulations_data.items():

    print(f"\nProcessing system: {system_name}")

    try:

        # ------------------------------------------
        # Load trajectory
        # ------------------------------------------

        traj = md.load(
            system_info["traj"],
            top=system_info["topo"],
            stride=TRAJ_STRIDE,
        )

        print(
            f"Trajectory loaded: "
            f"{traj.n_frames} frames"
        )


        # ------------------------------------------
        # Select nucleic acid residues
        # ------------------------------------------

        nucleic_residues = sorted(

            [
                residue
                for residue in traj.topology.residues
                if residue.name in NUCLEIC_ACID_RESNAMES
            ],

            key=lambda residue: residue.resSeq,

        )

        if len(nucleic_residues) == 0:

            print(
                "[Warning] No nucleic acid residues found."
            )

            continue

        print(
            f"Nucleic acid residues: "
            f"{len(nucleic_residues)}"
        )


        # ------------------------------------------
        # Select ligand residue
        # ------------------------------------------

        ligand_residues = [

            residue
            for residue in traj.topology.residues
            if residue.name == LIGAND_RESNAME

        ]

        if len(ligand_residues) == 0:

            print(
                f"[Warning] Ligand residue "
                f"{LIGAND_RESNAME} not found."
            )

            continue

        ligand_residue = ligand_residues[0]

        print(
            f"Ligand residue found: "
            f"{ligand_residue.name}"
            f"{ligand_residue.resSeq}"
        )


        # ------------------------------------------
        # Calculate contact frequency
        # ------------------------------------------

        (
            residue_labels_original_order,
            contact_frequencies,
        ) = calculate_contact_frequency(

            traj=traj,

            nucleic_residues=nucleic_residues,

            ligand_residue=ligand_residue,

            cutoff_nm=CONTACT_CUTOFF_NM,

        )


        # ------------------------------------------
        # Build residue reorder standard once
        # ------------------------------------------

        if reordered_labels is None:

            reordered_labels, group_sizes = reorder_residue_labels(
                residue_labels_original_order
            )

            print(
                "Residue order standard created."
            )


        # ------------------------------------------
        # Reorder contact frequency data
        # ------------------------------------------

        current_data_map = dict(
            zip(
                residue_labels_original_order,
                contact_frequencies,
            )
        )

        reordered_frequencies = [

            current_data_map.get(label, 0)

            for label in reordered_labels

        ]

        results[system_name] = reordered_frequencies

        print(
            f"System {system_name} completed."
        )

    except Exception as error:

        print(
            f"[ERROR] Failed to process system "
            f"{system_name}: {error}"
        )


# ==================================================
# Part 4. Plot Contact Frequency
# ==================================================

print("\nGenerating contact frequency plot ...")


if not results or reordered_labels is None:

    print(
        "[ERROR] No valid contact frequency data."
    )

else:

    num_systems = len(results)
    num_residues = len(reordered_labels)

    total_bar_width = 0.8
    single_bar_width = (
        total_bar_width / num_systems
    )

    bar_positions = np.arange(
        num_residues
    )

    fig, ax = plt.subplots(
        figsize=FIG_SIZE,
        dpi=FIG_DPI,
    )


    # ----------------------------------------------
    # Draw grouped bar plot
    # ----------------------------------------------

    for system_id, system_name in enumerate(results.keys()):

        offset = (
            system_id
            - (num_systems - 1) / 2
        ) * single_bar_width

        positions = (
            bar_positions
            + offset
        )

        ax.bar(

            positions,

            results[system_name],

            width=single_bar_width,

            label=simulations_data[system_name]["label"],

            color=simulations_data[system_name]["color"],

            edgecolor="black",

            linewidth=0.5,

        )


    # ----------------------------------------------
    # Draw background regions
    # ----------------------------------------------

    g1_end = group_sizes["g1"]
    g2_end = g1_end + group_sizes["g2"]
    g3_end = g2_end + group_sizes["g3"]
    at_end = g3_end + group_sizes["at"]

    ax.axvspan(
        -0.5,
        g1_end - 0.5,
        facecolor="skyblue",
        alpha=0.15,
        zorder=-10,
    )

    ax.axvspan(
        g1_end - 0.5,
        g2_end - 0.5,
        facecolor="crimson",
        alpha=0.15,
        zorder=-10,
    )

    ax.axvspan(
        g2_end - 0.5,
        g3_end - 0.5,
        facecolor="tomato",
        alpha=0.15,
        zorder=-10,
    )

    ax.axvspan(
        g3_end - 0.5,
        at_end - 0.5,
        facecolor="royalblue",
        alpha=0.15,
        zorder=-10,
    )


    # ----------------------------------------------
    # Draw separator lines
    # ----------------------------------------------

    separator_positions = [
        g1_end - 0.5,
        g2_end - 0.5,
        g3_end - 0.5,
        at_end - 0.5,
    ]

    for position in separator_positions:

        if position < num_residues - 0.5:

            ax.axvline(
                x=position,
                color="black",
                linestyle="--",
                linewidth=1.5,
            )


    # ----------------------------------------------
    # Figure formatting
    # ----------------------------------------------

    ax.set_xlabel(
        "Nucleic Acid Residue",
        fontsize=22,
    )

    ax.set_ylabel(
        "Contact Frequency with Probe",
        fontsize=22,
    )

    ax.set_title(
        f"Comparison of Contact Frequencies "
        f"(Cutoff < {CONTACT_CUTOFF_NM} nm)",
        fontsize=24,
    )

    ax.set_xticks(
        bar_positions
    )

    ax.set_xticklabels(
        reordered_labels,
        rotation=60,
        ha="right",
        fontsize=20,
    )

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=20,
    )

    ax.set_ylim(
        Y_MIN,
        Y_MAX,
    )

    ax.set_xlim(
        -0.5,
        num_residues - 0.5,
    )

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.7,
    )

    ax.legend(
        fontsize=20,
    )

    plt.tight_layout()
    plt.show()

    print("Contact frequency plot completed.")